# Project 07 – Text Summarizer

**Description:** Build an extractive text summarizer using word frequency scoring.

**Learning Goals:**
- Understand tokenization (splitting text into words and sentences)
- Learn why stop words are removed in NLP pipelines
- Score and rank sentences using word frequency
- Distinguish extractive vs abstractive summarization

**Estimated time:** 45–60 minutes

---

## Step 1 — Install & Import Libraries

We use **NLTK** (Natural Language Toolkit), one of the most popular Python NLP libraries.
It provides tools for tokenization, stop words, and more.

In [ ]:
# Install dependencies (run once)
# !pip install nltk numpy

import re
import heapq
import nltk
from collections import defaultdict

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

print('Libraries loaded successfully!')

## Step 2 — Load Sample Text

We will use a short article about Artificial Intelligence as our test document.
You can replace this with any text you like!

In [ ]:
article = """
Artificial intelligence (AI) is intelligence demonstrated by machines,
as opposed to the natural intelligence displayed by animals and humans.
AI research has been defined as the field of study of intelligent agents,
which refers to any system that perceives its environment and takes actions
that maximize its chance of achieving its goals.

The term 'artificial intelligence' had previously been used to describe
machines that mimic and display human cognitive skills associated with the
human mind, such as learning and problem solving. This definition has since
been rejected by major AI researchers who now describe AI in terms of
rationality and acting rationally, which does not limit how intelligence
can be articulated.

AI applications include advanced web search engines, recommendation systems
(such as those used by YouTube, Amazon, and Netflix), understanding human speech
(such as Siri and Alexa), self-driving cars, generative AI tools (such as
ChatGPT and AI art), automated decision-making, and competing at the highest
level in strategic game systems such as chess and Go.

As machines become increasingly capable, tasks considered to require intelligence
are often removed from the definition of AI, a phenomenon known as the AI effect.
For instance, optical character recognition is frequently excluded from things
considered to be AI, having become a routine technology.

Artificial intelligence was founded as an academic discipline in 1956, and in the
years since it has experienced several waves of optimism, followed by disappointment
and the loss of funding, followed by new approaches, success, and renewed funding.
AI research has tried and discarded many different approaches, including simulating
the brain, modelling human problem solving, formal logic, large databases of
knowledge, and imitating animal behaviour.
"""

sentences = sent_tokenize(article)
print(f'Article has {len(article.split())} words and {len(sentences)} sentences.')

## Step 3 — Tokenization & Stop Words

**Tokenization** is the process of splitting text into individual units (tokens).

**Stop words** are very common words (like *the*, *is*, *in*) that appear in almost
every sentence but carry very little meaning. We remove them so our frequency
scores reflect *meaningful* words.

In [ ]:
stop_words = set(stopwords.words('english'))

# Show a sample of stop words
sample_stops = sorted(list(stop_words))[:20]
print('Sample stop words:', sample_stops)

# Tokenize the article into words
all_words = word_tokenize(article.lower())
meaningful_words = [w for w in all_words if w.isalpha() and w not in stop_words]

print(f'\nTotal tokens:       {len(all_words)}')
print(f'After stop removal: {len(meaningful_words)}')

## Step 4 — Word Frequency Scoring

Count how many times each meaningful word appears, then normalize the counts
to a 0–1 scale by dividing by the maximum count.

In [ ]:
def compute_word_frequencies(text):
    """Return a normalized word frequency dict (stop words excluded)."""
    stop_words = set(stopwords.words('english'))
    # Clean text first
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    words = word_tokenize(cleaned.lower())

    word_freq = defaultdict(int)
    for word in words:
        if word.isalpha() and word not in stop_words:
            word_freq[word] += 1

    # Normalize
    max_freq = max(word_freq.values()) if word_freq else 1
    return {w: c / max_freq for w, c in word_freq.items()}


word_freq = compute_word_frequencies(article)

# Display top 10 words
top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]
print(f'{'Word':<20} {'Normalized Score':>16}')
print('-' * 38)
for word, score in top_words:
    print(f'{word:<20} {score:>16.3f}')

## Step 5 — Sentence Scoring

For each sentence, add up the normalized frequency scores of all its
meaningful words. Sentences whose words appear most often in the text
tend to capture the main topics.

In [ ]:
def score_sentences(sentences, word_freq):
    """Score each sentence by summing word frequencies."""
    sentence_scores = {}
    for sentence in sentences:
        words = word_tokenize(sentence.lower())
        word_count = len([w for w in words if w.isalpha()])
        # Only consider sentences of reasonable length
        if 5 <= word_count <= 30:
            score = sum(word_freq.get(w, 0) for w in words if w.isalpha())
            sentence_scores[sentence] = score
    return sentence_scores


sentence_scores = score_sentences(sentences, word_freq)

print('Top 5 scored sentences:\n')
for sent, score in sorted(sentence_scores.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'Score {score:.2f}: {sent[:90]}...')
    print()

## Step 6 — Generate the Summary

Pick the top N sentences by score, then put them back in the order they
appeared in the original text so the summary reads naturally.

In [ ]:
def summarize(text, num_sentences=3):
    """Generate an extractive summary of `text`."""
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text

    word_freq = compute_word_frequencies(text)
    sentence_scores = score_sentences(sentences, word_freq)

    top_sentences = heapq.nlargest(num_sentences, sentence_scores, key=sentence_scores.get)
    # Preserve reading order
    return ' '.join(s for s in sentences if s in top_sentences)


summary = summarize(article, num_sentences=3)
print('=== 3-Sentence Summary ===')
print(summary)

## Step 7 — Compare Summary Lengths

In [ ]:
for n in [2, 3, 4]:
    s = summarize(article, num_sentences=n)
    print(f'--- {n}-sentence summary ({len(s.split())} words) ---')
    print(s)
    print()

## Try It Yourself!

1. Paste any news article or Wikipedia paragraph into the cell below as `my_text`.
2. Try different values of `num_sentences` (1, 2, 3, 5).
3. What happens if your text is very short?
4. Try printing `sentence_scores` to see which sentences scored highest — do they make sense as the "main point"?

In [ ]:
my_text = """
Paste any article here and see how the summarizer performs!
The more sentences the article has, the better the summary will be.
Try copying a few paragraphs from any Wikipedia article.
"""

print(summarize(my_text, num_sentences=2))

## Summary — Concepts Learned

| Concept | Plain English Explanation |
|---------|---------------------------|
| Tokenization | Splitting text into individual words or sentences |
| Stop words | Common words (the, is, in) that add noise; we remove them |
| Word frequency | How often a word appears — more = more important |
| Normalization | Scaling scores to 0–1 so they're comparable |
| Sentence scoring | Summing word frequencies per sentence to rank importance |
| Extractive summarization | Selecting existing sentences (not generating new ones) |
| Abstractive summarization | Generating new sentences (requires deep learning / LLMs) |
| `heapq.nlargest` | Python built-in to efficiently find the top N items |